# 09 · Refinement Network 학습 (Phase 7-B)

**목적**: SC-FEGAN 출력의 artifact(노이즈, 경계 블러)를 작은 U-Net + L1+Perceptual로 정제.

**환경**: Colab T4 GPU. PyTorch native (TF 없음).

**산출물**: `project/refinement/checkpoints/refinement_best.pt`

In [ ]:
import os, sys
IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = '/content/cv-final'
    if not os.path.exists(REPO):
        # private repo → Colab Secrets의 GH_TOKEN 사용
        from google.colab import userdata
        token = userdata.get('GH_TOKEN')
        !git clone https://{token}@github.com/keonU206/cv-final.git $REPO
    %cd $REPO
    !pip install -q segmentation_models_pytorch torchvision pyyaml opencv-python Pillow
else:
    os.chdir(os.path.abspath('..'))
print(f'CWD: {os.getcwd()}')

## 1. 데이터셋 준비 (CelebAMask-HQ images)

U-Net 학습에서 이미 다운받은 폴더를 재사용. 없으면 새로 다운받음.

In [ ]:
from pathlib import Path
DATA_DIR = Path('/content/data/celebamask/images')
if not DATA_DIR.exists():
    print('데이터셋 다운로드 (HuggingFace mattymchen/celeba-hq)...')
    !pip install -q datasets
    from datasets import load_dataset
    ds = load_dataset('mattymchen/celeba-hq', split='train', streaming=False)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    import cv2, numpy as np
    for i, item in enumerate(ds):
        if i >= 2000: break  # subset_size에 맞춰
        img = np.array(item['image'])
        img = cv2.resize(img, (512, 512))
        cv2.imwrite(str(DATA_DIR / f'{i:05d}.jpg'), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    print(f'✅ {len(list(DATA_DIR.glob("*.jpg")))} images saved')
else:
    print(f'기존 데이터셋 사용: {len(list(DATA_DIR.glob("*.jpg")))} images')

## 2. Synthetic noise 효과 미리 확인

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt
from project.refinement.data import add_synthetic_noise

samples = sorted(DATA_DIR.glob('*.jpg'))[:3]
fig, axes = plt.subplots(3, 2, figsize=(8, 12))
for i, p in enumerate(samples):
    clean = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    noisy = add_synthetic_noise(clean, gaussian_sigma=8.0, jpeg_quality=60, seed=i)
    axes[i, 0].imshow(clean); axes[i, 0].set_title('Clean (target)')
    axes[i, 1].imshow(noisy); axes[i, 1].set_title('+ synthetic noise (input)')
    for ax in axes[i]:
        ax.axis('off')
plt.tight_layout(); plt.show()

## 3. 학습 실행

In [ ]:
from project.refinement.train import train
result = train(config_path='project/refinement/config.yaml')
print('\n=== Result ===')
for k, v in result.items():
    if k != 'history':
        print(f'{k}: {v}')

## 4. 학습 결과 시각화

In [ ]:
import torch
from project.refinement.model import build_refinement_wrapper
from project.refinement.data import SyntheticRefinementDataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = build_refinement_wrapper(encoder_weights=None).to(device)
model.load_state_dict(torch.load(result['best_ckpt_path'], map_location=device))
model.eval()

ds = SyntheticRefinementDataset(image_dir=DATA_DIR, size=512, subset_size=4, seed=99)
fig, axes = plt.subplots(4, 3, figsize=(12, 16))
with torch.no_grad():
    for i in range(4):
        s = ds[i]
        inp = s['input'].unsqueeze(0).to(device)
        pred = model(inp)[0].cpu()
        for j, (img, title) in enumerate([
            (s['input'], 'Noisy input'),
            (pred, 'Refined'),
            (s['target'], 'GT clean'),
        ]):
            arr = ((img.clamp(-1, 1) + 1) * 127.5).byte().permute(1, 2, 0).numpy()
            axes[i, j].imshow(arr); axes[i, j].set_title(title); axes[i, j].axis('off')
plt.tight_layout()
plt.savefig('project/refinement/samples/final_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# checkpoint를 Drive로 복사 (Colab 세션 종료에도 보존)
if IS_COLAB:
    DRIVE_CKPT = '/content/drive/MyDrive/cv-final-ckpts/refinement_best.pt'
    Path(DRIVE_CKPT).parent.mkdir(parents=True, exist_ok=True)
    !cp {result['best_ckpt_path']} {DRIVE_CKPT}
    print(f'✅ Drive 백업: {DRIVE_CKPT}')